In [1]:
import json  # permet de manipuler du JSON en Python

def compute_score(solution_str, ground_truth):
    """
    solution_str : texte généré par le LLM
    ground_truth : dictionnaire attendu
    """

    # Nettoie la réponse (supprime espaces et balises markdown éventuelles)
    clean_str = solution_str.strip().replace("```json", "").replace("```", "")

    reward = 0.0  # initialisation de la récompense

    try:
        # Tente de convertir le texte en JSON Python (dict)
        parsed_json = json.loads(clean_str)

        # Vérifie si toutes les clés attendues sont présentes
        if all(key in parsed_json for key in ground_truth):

            # Vérifie si le contenu est exactement correct
            if parsed_json == ground_truth:
                reward = 1.0  # réponse parfaite
            else:
                reward = 0.5  # JSON valide mais contenu partiellement correct

        else:
            reward = 0.5  # JSON valide mais incomplet

    except json.JSONDecodeError:
        # Si le modèle n’a pas généré un JSON valide
        reward = 0.0

    return reward  # retourne la récompense finale

In [2]:
# importe le trainer PPO de VERL
from verl.trainer.ppo.ray_trainer import RayPPOTrainer, Role

# workers utilisés pour Actor et Critic
from verl.workers.fsdp_workers import ActorRolloutRefWorker, CriticWorker

# importe ta fonction de récompense
from json_reward import compute_score

# ================= CONFIGURATION =================

# configuration simplifiée (tu peux adapter selon GPU)
config = {
    "train_batch_size": 2,  # nombre d'exemples par batch
    "ppo_epochs": 1,        # nombre d'itérations PPO
}

# tokenizer (simplifié ici, normalement depuis HuggingFace)
tokenizer = None  # placeholder (VERL le gère souvent automatiquement)

# ================= ROLES =================

# mapping des rôles vers les workers
role_worker_mapping = {

    # Actor : modèle qui génère les réponses
    Role.ActorRollout: ActorRolloutRefWorker,

    # RefPolicy : modèle de référence pour KL penalty
    Role.RefPolicy: ActorRolloutRefWorker,
}

# ================= CHOIX ALGORITHME =================

# --- DECOMMENTE POUR PPO ---
# si activé → ajoute un Critic → PPO complet
# sinon → REINFORCE

# role_worker_mapping[Role.Critic] = CriticWorker

# ================= REWARD =================

def reward_manager(data_source):
    """
    data_source contient les réponses générées
    retourne la fonction de récompense
    """
    return compute_score

# ================= TRAINER =================

trainer = RayPPOTrainer(
    config=config,                       # configuration globale
    tokenizer=tokenizer,                # tokenizer du modèle
    role_worker_mapping=role_worker_mapping,  # définit Actor/Critic
    reward_fn=reward_manager            # fonction de récompense
)

# ================= EXECUTION =================

print("Initialisation des workers GPU...")
trainer.init_workers()  # démarre les processus distribués

print("Début entraînement RL...")
trainer.fit()  # lance la boucle d’apprentissage

ModuleNotFoundError: No module named 'verl'

In [5]:
%cd verl

/content/verl


In [1]:
!pip install vllm

In [2]:
!pip install -e .

Obtaining file:///content
ERROR: file:///content does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


In [3]:
%%writefile json_reward.py

import json  # bibliothèque pour manipuler JSON

def compute_score(solution_str, ground_truth):
    """
    solution_str : sortie du LLM
    ground_truth : réponse correcte (dict)
    """

    # Nettoyage du texte
    clean_str = solution_str.strip().replace("```json", "").replace("```", "")

    reward = 0.0  # initialisation

    try:
        # conversion texte → dictionnaire
        parsed_json = json.loads(clean_str)

        # vérifier les clés
        if all(key in parsed_json for key in ground_truth):

            # vérifier contenu exact
            if parsed_json == ground_truth:
                reward = 1.0  # parfait
            else:
                reward = 0.5  # partiellement correct
        else:
            reward = 0.5  # incomplet

    except json.JSONDecodeError:
        # pas du JSON valide
        reward = 0.0

    return reward  # retourne score

Overwriting json_reward.py


In [3]:
%%writefile trainer.py

# importe le trainer PPO
from verl.trainer.ppo.ray_trainer import RayPPOTrainer, Role

# workers Actor / Critic
from verl.workers.fsdp_workers import ActorRolloutRefWorker, CriticWorker

# fonction de récompense
from json_reward import compute_score

# ================= CONFIG =================

config = {
    "train_batch_size": 2,  # petit batch pour Colab
    "ppo_epochs": 1         # nombre d’itérations
}

tokenizer = None  # simplifié

# ================= ROLES =================

role_worker_mapping = {
    Role.ActorRollout: ActorRolloutRefWorker,  # modèle principal
    Role.RefPolicy: ActorRolloutRefWorker,     # modèle référence
}

# ================= PPO =================

# Décommente pour activer PPO
role_worker_mapping[Role.Critic] = CriticWorker

# ================= REWARD =================

def reward_manager(data_source):
    return compute_score  # lie la fonction reward

# ================= TRAINER =================

trainer = RayPPOTrainer(
    config=config,
    tokenizer=tokenizer,
    role_worker_mapping=role_worker_mapping,
    reward_fn=reward_manager
)

print("Initialisation des workers...")
trainer.init_workers()

print("Début entraînement...")
trainer.fit()

Writing trainer.py


In [4]:
!python trainer.py

Traceback (most recent call last):
  File "/content/trainer.py", line 3, in <module>
    from verl.trainer.ppo.ray_trainer import RayPPOTrainer, Role
ModuleNotFoundError: No module named 'verl.trainer'
